# Keep-alive

In [ ]:
%%javascript
function ClickConnect(){
    console.log("Keeping alive");
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)

<IPython.core.display.Javascript object>

# Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Install and Imports

In [ ]:
!pip install shap --quiet

import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, GRU, Dense, Dropout,
    Bidirectional, Multiply, Permute,
    RepeatVector, Flatten, Activation,
    Lambda
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.20.0
GPU: []


# Configuration

In [ ]:
# Options: 'baseline', 'bilstm', 'attention', 'gru'
MODEL_TYPE = 'gru'
# Paths
CSV_PATH    = '/content/drive/MyDrive/DMIF/master_dataset_clean_80.csv'
LOCAL_CSV   = '/content/master_dataset.csv'
MODELS_DIR  = '/content/drive/MyDrive/DMIF/models'
os.makedirs(MODELS_DIR, exist_ok=True)

# Model save paths
SAVE_PATHS = {
    'baseline'  : f'{MODELS_DIR}/lstm_baseline.keras',
    'bilstm'    : f'{MODELS_DIR}/lstm_bilstm.keras',
    'attention' : f'{MODELS_DIR}/lstm_attention.keras',
    'gru'       : f'{MODELS_DIR}/lstm_gru.keras'
}
SAVE_PATH = SAVE_PATHS[MODEL_TYPE]

# Training config
SEQ_LEN      = 30
BATCH_SIZE   = 64
EPOCHS       = 50
DROPOUT      = 0.3
LSTM_UNITS   = [64, 32]

# Class weights to fix imbalance (UP 34.6% / DOWN 65.4%)
CLASS_WEIGHT = {0: 1.0, 1: 1.89}

# LSTM features — 30 features
LSTM_FEATURES = [
    'Open', 'High', 'Low', 'Close', 'Log_Volume',
    'Daily_Return', 'Log_Return', 'HL_Range', 'OC_Range',
    'MA_24', 'MA_30', 'MA_500',
    'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist',
    'BB_Upper', 'BB_Lower', 'BB_Middle', 'BB_Width',
    'Volatility_10', 'SAR', 'SAR_Trend',
    'Fib_0236', 'Fib_0382', 'Fib_05', 'Fib_0618', 'Fib_0786',
    'Volume_MA_10', 'Volume_Ratio'
]

print(f"Model type : {MODEL_TYPE}")
print(f"Save path  : {SAVE_PATH}")
print(f"Features   : {len(LSTM_FEATURES)}")
print(f"Input shape: ({SEQ_LEN}, {len(LSTM_FEATURES)})")

Model type : gru
Save path  : /content/drive/MyDrive/DMIF/models/lstm_gru.keras
Features   : 30
Input shape: (30, 30)


# Load and Preprocess Data

In [ ]:
import shutil

print("Copying CSV to local disk...")
shutil.copy(CSV_PATH, LOCAL_CSV)

df = pd.read_csv(LOCAL_CSV, parse_dates=['Date'])
df = df.sort_values(['Company_Code', 'Date']).reset_index(drop=True)

print(f"Loaded: {df.shape[0]} rows, {df['Company_Code'].nunique()} companies")

# ── Build sequences per company ───────────────────────────────
all_X, all_y = [], []
scalers = {}

companies = sorted(df['Company_Code'].unique())

for company in companies:
    company_df = df[df['Company_Code'] == company].copy()
    company_df = company_df.sort_values('Date').reset_index(drop=True)

    # Drop rows where any feature is NaN
    company_df = company_df.dropna(subset=LSTM_FEATURES + ['Target'])

    if len(company_df) < SEQ_LEN + 1:
        continue

    # Scale features — fit on training portion only (70%)
    train_end = int(len(company_df) * 0.7)
    scaler = MinMaxScaler()
    scaler.fit(company_df[LSTM_FEATURES].iloc[:train_end])
    scalers[company] = scaler

    scaled = scaler.transform(company_df[LSTM_FEATURES])
    targets = company_df['Target'].values

    # Build sliding windows
    for i in range(SEQ_LEN, len(company_df) - 1):
        all_X.append(scaled[i-SEQ_LEN:i])
        all_y.append(targets[i])

all_X = np.array(all_X, dtype=np.float32)
all_y = np.array(all_y, dtype=np.float32)

print(f"\nTotal sequences : {len(all_X)}")
print(f"Input shape     : {all_X.shape}")
print(f"UP  labels      : {all_y.sum():.0f} ({all_y.mean()*100:.1f}%)")
print(f"DOWN labels     : {(1-all_y).sum():.0f} ({(1-all_y).mean()*100:.1f}%)")

Copying CSV to local disk...
Loaded: 323998 rows, 80 companies

Total sequences : 321518
Input shape     : (321518, 30, 30)
UP  labels      : 112904 (35.1%)
DOWN labels     : 208614 (64.9%)


# Train/Val/Test Split

In [ ]:
n = len(all_X)
train_end = int(n * 0.70)
val_end   = int(n * 0.80)

X_tr = all_X[:train_end]
y_tr = all_y[:train_end]
X_va = all_X[train_end:val_end]
y_va = all_y[train_end:val_end]
X_te = all_X[val_end:]
y_te = all_y[val_end:]

print(f"Train : {X_tr.shape} | UP: {y_tr.mean()*100:.1f}%")
print(f"Val   : {X_va.shape} | UP: {y_va.mean()*100:.1f}%")
print(f"Test  : {X_te.shape} | UP: {y_te.mean()*100:.1f}%")

Train : (225062, 30, 30) | UP: 34.3%
Val   : (32152, 30, 30) | UP: 37.6%
Test  : (64304, 30, 30) | UP: 36.8%


# Model Definitions

In [ ]:
def build_baseline_lstm(seq_len, n_features):
    """Original 2-layer LSTM — matches paper architecture"""
    inp = Input(shape=(seq_len, n_features))
    x   = LSTM(64, return_sequences=True)(inp)
    x   = Dropout(0.3)(x)
    x   = LSTM(32, return_sequences=False)(x)
    x   = Dropout(0.3)(x)
    x   = Dense(32, activation='relu')(x)
    x   = Dropout(0.15)(x)
    out = Dense(1, activation='sigmoid')(x)
    return Model(inp, out, name='LSTM_Baseline')


def build_bilstm(seq_len, n_features):
    """Bidirectional LSTM — reads sequence forward and backward"""
    inp = Input(shape=(seq_len, n_features))
    x   = Bidirectional(LSTM(64, return_sequences=True))(inp)
    x   = Dropout(0.3)(x)
    x   = Bidirectional(LSTM(32, return_sequences=False))(x)
    x   = Dropout(0.3)(x)
    x   = Dense(32, activation='relu')(x)
    x   = Dropout(0.15)(x)
    out = Dense(1, activation='sigmoid')(x)
    return Model(inp, out, name='LSTM_Bidirectional')


def build_attention_lstm(seq_len, n_features):
    """LSTM with attention mechanism"""
    inp = Input(shape=(seq_len, n_features))
    x   = LSTM(64, return_sequences=True)(inp)
    x   = Dropout(0.3)(x)
    x   = LSTM(32, return_sequences=True)(x)
    x   = Dropout(0.3)(x)

    # Attention layer
    attention = Dense(1, activation='tanh')(x)
    attention = Flatten()(attention)
    attention = Activation('softmax')(attention)
    attention = RepeatVector(32)(attention)
    attention = Permute([2, 1])(attention)

    # Apply attention weights
    x   = Multiply()([x, attention])
    x   = Lambda(lambda z: tf.reduce_sum(z, axis=1))(x)
    x   = Dense(32, activation='relu')(x)
    x   = Dropout(0.15)(x)
    out = Dense(1, activation='sigmoid')(x)
    return Model(inp, out, name='LSTM_Attention')


def build_gru(seq_len, n_features):
    """GRU — faster alternative to LSTM"""
    inp = Input(shape=(seq_len, n_features))
    x   = GRU(64, return_sequences=True)(inp)
    x   = Dropout(0.3)(x)
    x   = GRU(32, return_sequences=False)(x)
    x   = Dropout(0.3)(x)
    x   = Dense(32, activation='relu')(x)
    x   = Dropout(0.15)(x)
    out = Dense(1, activation='sigmoid')(x)
    return Model(inp, out, name='GRU_Baseline')


# Build selected model
builders = {
    'baseline'  : build_baseline_lstm,
    'bilstm'    : build_bilstm,
    'attention' : build_attention_lstm,
    'gru'       : build_gru
}

model = builders[MODEL_TYPE](SEQ_LEN, len(LSTM_FEATURES))
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "GRU_Baseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 30, 30)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 30, 64)         │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,929 (113.00 KB)

 Trainable params: 28,929 (113.00 KB)

 Non-trainable params: 0 (0.00 B)

# Train

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        verbose=1
    ),
    ModelCheckpoint(
        SAVE_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print(f"Training {MODEL_TYPE} model...")
print(f"Class weights: {CLASS_WEIGHT}")

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=CLASS_WEIGHT,
    callbacks=callbacks,
    verbose=1
)

print(f"\n✓ Training complete")
print(f"✓ Best model saved to: {SAVE_PATH}")

Training gru model...
Class weights: {0: 1.0, 1: 1.89}
Epoch 1/50
3516/3517 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.5135 - loss: 0.9048
Epoch 1: val_accuracy improved from None to 0.49011, saving model to /content/drive/MyDrive/DMIF/models/lstm_gru.keras

Epoch 1: finished saving model to /content/drive/MyDrive/DMIF/models/lstm_gru.keras
3517/3517 ━━━━━━━━━━━━━━━━━━━━ 199s 54ms/step - accuracy: 0.5185 - loss: 0.9024 - val_accuracy: 0.4901 - val_loss: 0.6974 - learning_rate: 0.0010
Epoch 2/50
3516/3517 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.5316 - loss: 0.8991
Epoch 2: val_accuracy did not improve from 0.49011
3517/3517 ━━━━━━━━━━━━━━━━━━━━ 186s 50ms/step - accuracy: 0.5339 - loss: 0.8982 - val_accuracy: 0.4824 - val_loss: 0.6959 - learning_rate: 0.0010
Epoch 3/50
3516/3517 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.5262 - loss: 0.8968
Epoch 3: val_accuracy did not improve from 0.49011
3517/3517 ━━━━━━━━━━━━━━━━━━━━ 203s 50ms/step - accuracy: 0.5302 - loss: 0.8960 

# Evaluate

In [ ]:
def build_attention_lstm(seq_len, n_features):
    """LSTM with attention mechanism — no Lambda layer"""
    inp = Input(shape=(seq_len, n_features))
    x   = LSTM(64, return_sequences=True)(inp)
    x   = Dropout(0.3)(x)
    x   = LSTM(32, return_sequences=True)(x)
    x   = Dropout(0.3)(x)

    # Attention — using Dense instead of Lambda
    attention = Dense(1, activation='tanh')(x)      # (batch, seq, 1)
    attention = Flatten()(attention)                 # (batch, seq)
    attention = Activation('softmax')(attention)     # (batch, seq)
    attention = tf.keras.layers.Reshape((seq_len, 1))(attention)  # (batch, seq, 1)

    # Apply attention weights by multiplying and summing
    x   = tf.keras.layers.Multiply()([x, attention])  # (batch, seq, 32)
    x   = tf.keras.layers.GlobalAveragePooling1D()(x)  # (batch, 32)
    x   = Dense(32, activation='relu')(x)
    x   = Dropout(0.15)(x)
    out = Dense(1, activation='sigmoid')(x)
    return Model(inp, out, name='LSTM_Attention')

# Plot Training History

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'],     label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title(f'{MODEL_TYPE} — Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history.history['loss'],     label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title(f'{MODEL_TYPE} — Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plot_path = f'{MODELS_DIR}/{MODEL_TYPE}_history.png'
plt.savefig(plot_path)
plt.show()
print(f"Plot saved: {plot_path}")

NameError: name 'plt' is not defined

# Save Results Summary

In [ ]:
import json

results = {
    'model_type'     : MODEL_TYPE,
    'train_accuracy' : float(tr_acc),
    'val_accuracy'   : float(va_acc),
    'test_accuracy'  : float(te_acc),
    'train_samples'  : int(len(X_tr)),
    'val_samples'    : int(len(X_va)),
    'test_samples'   : int(len(X_te)),
    'features'       : len(LSTM_FEATURES),
    'seq_len'        : SEQ_LEN,
    'class_weight'   : CLASS_WEIGHT,
    'epochs_trained' : len(history.history['accuracy'])
}

results_path = f'{MODELS_DIR}/{MODEL_TYPE}_results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved: {results_path}")
print(json.dumps(results, indent=2))

Results saved: /content/drive/MyDrive/DMIF/models/bilstm_results.json
{
  "model_type": "bilstm",
  "train_accuracy": 0.5674347513129715,
  "val_accuracy": 0.526343617815377,
  "test_accuracy": 0.5450671808907689,
  "train_samples": 225062,
  "val_samples": 32152,
  "test_samples": 64304,
  "features": 30,
  "seq_len": 30,
  "class_weight": {
    "0": 1.0,
    "1": 1.89
  },
  "epochs_trained": 8
}


# Match created images

In [ ]:
import os

CHARTS_DIR = '/content/drive/MyDrive/DMIF/charts_final'

cs_count = 0
hm_count = 0

for company in os.listdir(CHARTS_DIR):
    company_dir = os.path.join(CHARTS_DIR, company)
    if not os.path.isdir(company_dir):
        continue
    for f in os.listdir(company_dir):
        if 'candlestick' in f:
            cs_count += 1
        elif 'heatmap' in f:
            hm_count += 1

print(f"Candlestick : {cs_count}")
print(f"Heatmap     : {hm_count}")
print(f"Match       : {'✅ Yes' if cs_count == hm_count else '❌ No'}")

Candlestick : 64338
Heatmap     : 64338
Match       : ✅ Yes
